In [7]:
from src_main.api_client import BaseClient
from src_main.market_data import MarketDataClient
from src_main.db_manager import DatabaseManager
from src_main.api_job_logger import ApiJobLogger
from datetime import timedelta, datetime, timezone

client = MarketDataClient()
db = DatabaseManager()
logger = ApiJobLogger(db)

figi = "BBG004730N88"
interval = "15min"
to_dt = datetime.now(timezone.utc)
from_dt = to_dt - timedelta(days=1)

db_table_col = {
    'id': 'BIGSERIAL PRIMARY KEY',
    'endpoint': 'VARCHAR(255) NOT NULL',
    'request_payload': 'JSONB',        
    'response_data': 'JSONB', 
    'status_code': 'INTEGER', 
    'loaded_at': 'TIMESTAMPTZ DEFAULT NOW()'
}

db.create_schema(schema_name='raw')
db.create_table(table_name='raw.extract_api_jobs', columns_type=db_table_col)

(response, status_code), endpoint, payload = client.get_candles(figi, from_dt, to_dt, interval)

logger.save_job(
    endpoint=endpoint,
    request_payload=payload,
    response_data=response,
    status_code=status_code
)

200